# Which agencies have accomodations with rent below 1000 euro and for which rent subsidy is possible in their catalog? 

In [ ]:
%%sql
SELECT * FROM Agency 
JOIN Accommodation ON Agency.agency_id = Accommodation.agency_id
WHERE Accommodation.rent_price < 1000
AND Accommodation.subsidy_possible = 'yes';

# Which agency has the cheapest properties? Comparison of average property rent, maximum property rent, minimum property rent and total property rent per agecy.

In [ ]:
%%sql

SELECT name, 
AVG(Accommodation.rent_price) AS Avg_property_rent, 
MAX(Accommodation.rent_price) AS MAx_property_rent, 
MIN(Accommodation.rent_price) AS Min_property_rent ,  
SUM(Accommodation.rent_price) AS Total_property_rent
FROM Agency 
JOIN Accommodation 
ON Agency.agency_id = Accommodation.agency_id
GROUP BY name;

# How many acomodations are above and below cities' average?

In [ ]:
%%sql

WITH CityAverage AS (
SELECT city.city_code, AVG(Accommodation.rent_price) AS average_rent
FROM city JOIN Address
ON address.city_code = city.city_code
JOIN Accommodation ON Address.address_id = Accommodation.address_id
GROUP BY city.city_code
)
SELECT CityAverage.city_code, COUNT(
    CASE WHEN Accommodation.rent_price < CityAverage.average_rent THEN 1 END
) AS accomodation_below_average_rent,
COUNT(
    CASE WHEN Accommodation.rent_price > CityAverage.average_rent THEN 1 END
) AS accomodation_above_average_rent
FROM CityAverage
JOIN Address ON CityAverage.city_code = Address.city_code
JOIN Accommodation ON Address.Address_id = Accommodation.address_id
GROUP BY CityAverage.city_code;

# What acommodations fit students' requirements? Matching process for students.
Created by h-a-n-i-a

In [ ]:
%config SqlMagic.displaylimit = None

In [ ]:
%%sql
SELECT s.student_id, s.email AS student_email, s.max_rent AS student_budget, a.accommodation_id, a.rent_price, addr.street, addr.house_number, addr.city_code
FROM Student s
JOIN Address addr ON s.search_city_code = addr.city_code
JOIN Accommodation a ON addr.address_id = a.address_id
WHERE a.rent_price <= s.max_rent
ORDER BY s.student_id, a.rent_price ASC;

# Which students aren't matched yet?

In [ ]:
%%sql
SELECT s.student_id, s.email, s.search_city_code, s.max_rent
FROM Student s
LEFT JOIN Contract c ON s.student_id = c.student_id
WHERE c.contract_id IS NULL;

# Which landlords don't belong to any agency?

In [ ]:
%%sql
SELECT l.name, COUNT(a.accommodation_id) AS properties
FROM Landlord l
LEFT JOIN Accommodation a ON a.landlord_id = l.landlord_id
WHERE l.agency_id IS NULL
GROUP BY l.landlord_id, l.name;

# What are the ten cheapest places availible in Maastricht?
Created by natinajar8000

In [ ]:
%%sql
SELECT a.accommodation_id,
       a.rent_price,
       a.square_meters,
       a.accomodation_type,
       ad.street
FROM Accommodation a
JOIN Address ad ON a.address_id = ad.address_id
WHERE ad.city_code = 'MAASTRICHT'
ORDER BY a.rent_price ASC
LIMIT 10;

# Which accommodation does not have an active contract right now?

In [ ]:
%%sql
SELECT
    a.accommodation_id,
    ad.city_code,
    ad.street,
    ad.house_number,
    a.accomodation_type,
    a.rent_price
FROM Accommodation a
JOIN Address ad ON a.address_id = ad.address_id
WHERE NOT EXISTS (
    SELECT 1
    FROM Contract c
    WHERE c.accommodation_id = a.accommodation_id
      AND c.start_date <= CURDATE()
      AND (c.end_date >= CURDATE() OR c.end_date IS NULL)
)
ORDER BY a.rent_price;